In [7]:
import os
import pandas as pd
from typing import Optional, Dict

In [8]:
# constants
INPUT_PATH: str = "results.csv"        # change to your input CSV path
OUTPUT_PATH: Optional[str] = "resultsFix.csv"    # set to None to overwrite INPUT_PATH or set to desired output path
DEFAULT_COLUMNS = ["Qwen3-8B", "Meta-Llama-3.1-8B-Instruct"]

In [9]:
def remove_error_rows(in_path: str, out_path: str) -> Dict[str, int]:
    # Read CSV as strings and avoid converting empty strings to NaN
    df = pd.read_csv(in_path, dtype=str, keep_default_na=False)
    per_column_removed = {col: 0 for col in DEFAULT_COLUMNS}
    overall_mask = pd.Series(False, index=df.index)

    for col in DEFAULT_COLUMNS:
        if col in df.columns:
            s = df[col].fillna("").astype(str)
            col_mask = s.str.lstrip().str.lower().str.startswith("<error:")
            per_column_removed[col] = int(col_mask.sum())
            overall_mask = overall_mask | col_mask
        else:
            # Column missing: count remains 0
            per_column_removed[col] = 0

    removed_rows = int(overall_mask.sum())
    if removed_rows:
        df_clean = df.loc[~overall_mask].copy()
    else:
        df_clean = df

    df_clean.to_csv(out_path, index=False)
    # Return summary including total removed
    summary = {"total_removed_rows": removed_rows}
    summary.update(per_column_removed)
    return summary

In [10]:
def main() -> None:
    in_path = INPUT_PATH
    if OUTPUT_PATH:
        out_path = OUTPUT_PATH
        will_overwrite = False
    else:
        out_path = in_path + ".cleaned.tmp.csv"
        will_overwrite = True

    if not os.path.isfile(in_path):
        print(f"Input file not found: {in_path}")
        raise SystemExit(1)

    summary = remove_error_rows(in_path, out_path)

    if will_overwrite:
        os.replace(out_path, in_path)
        print(f"Overwrote {in_path} with cleaned CSV.")
    else:
        print(f"Wrote cleaned CSV to {out_path}.")

    print("Summary:")
    print(f"  total rows removed: {summary['total_removed_rows']}")
    for col in DEFAULT_COLUMNS:
        print(f"  {col}: {summary[col]} rows contained error and contributed to removal")

if __name__ == "__main__":
    main()

Wrote cleaned CSV to resultsFix.csv.
Summary:
  total rows removed: 36
  Qwen3-8B: 35 rows contained error and contributed to removal
  Meta-Llama-3.1-8B-Instruct: 36 rows contained error and contributed to removal
